# Tuning

In this homework, you'll tune some hyperparameters to try to improve the performance of a network. This will involve setting up a convolutional network for the CIFAR-10 dataset, setting up TensorBoard for logging, and then experimenting. I am _not_ going to be setting specific accuracy targets for different grades because there is too much randomness in the training process. Moreover, achieving high accuracy is easier if you have access to a lot of computational resources, so there are some equity issues with just grading by final model performance.

Instead, I'm going to ask you to explain your tuning _process_. That is, for each experiment you run (each set of hyperparameters you try), explain why you ran that experiment and what happened. Based on your observations, what changes did you make for the next run? As long as you have explained your reasoning and it corresponds to the principles we've talked about in class, you'll do fine. Be sure to set up your logging in a way that indicates the hyperparameter values used for each run.

**IMPORTANT: Please zip up and submit your TensorBoard log files with your homework. That will help me to see what you were looking at as you went through your tuning process.**

I'm also deliberately giving you no starter code for this homework. I understand that a lot of it will just be copy-paste from past classes/labs/homeworks but I still think there is some value in going from a blank document to a complete program.

In [1]:
# imports / setup

import torch
from torch import nn, optim
import numpy as np
import torchvision
torchvision.disable_beta_transforms_warning()
import torchvision.transforms.v2 as transforms
import torch.utils.tensorboard as tb
import datetime
import os

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

# data setup

transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ConvertImageDtype(),
])

cifar = torchvision.datasets.CIFAR10("../data/torch/cifar", download=True, transform=transform)
train_size = int(0.8 * len(cifar))
train_data, valid_data = torch.utils.data.random_split(cifar, [train_size, len(cifar) - train_size])

classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
print(len(cifar))

# mean = []
# for x, _ in cifar:
#     mean.append(torch.mean(x, dim=(1, 2)))
# mean = torch.stack(mean, dim=0).mean(dim=0)
# std = []
# for x, _ in cifar:
#     std.append(((x - mean[:,np.newaxis,np.newaxis]) ** 2).mean(dim=(1, 2)))
# std = torch.stack(std, dim=0).mean(dim=0).sqrt()

# I'm stealing these from the pytorch homework so I don't have to run the above code every time. It's the same I checked
cifar_mean = (0.4914, 0.4822, 0.4465)
cifar_std = (0.2470, 0.2435, 0.2616)
normalize = transforms.Normalize(cifar_mean, cifar_std)

# logging setup 
log_dir = 'tuning_hw_logs'

Files already downloaded and verified
50000


In [2]:
# define the model

class CNN(nn.Module):

    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 16, (7, 7), padding=3),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.AvgPool2d(kernel_size=(4, 4)),
            nn.Flatten(),
            nn.Linear(64*4*4, 10)
        )

    def forward(self, x):
        return self.model(x)

In [3]:
# train the model

def train(model_class=CNN, lr=1e-3, epochs=10, batch_size=64, reg=0):
 
    data_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
    valid_loader = torch.utils.data.DataLoader(valid_data, batch_size=batch_size, shuffle=False)

    train_accs = []
    valid_accs = []
    
    network=model_class().to(device)
    loss = nn.CrossEntropyLoss()
    opt = optim.SGD(network.parameters(), momentum=0.9, lr=lr, weight_decay=reg)

    # Build a name for each training run. In this case, the name has the format
    # input_size:hidden1_size:...:hiddenN_size-lr-XX-bs-XX-mom-XX
    # The first set of colon-separated integers encodes the MLP architecture while
    # lr, bs, and mom, capture the learning rate, batch size, and momentum respectively.
    name += '-lr-' + str(lr) + '-bs-' + str(batch_size) + '-eps-' + str(epochs)
    logger = tb.SummaryWriter(os.path.join(log_dir, name))
    global_step = 0

    for i in range(epochs):
        
        accs = []
        for batch_xs, batch_ys in data_loader:
            batch_xs = batch_xs.to(device)
            batch_ys = batch_ys.to(device) 
            batch_xs = normalize(batch_xs)
            preds = network(batch_xs)
            loss_val = loss(preds, batch_ys)
            acc = (preds.argmax(dim=1) == batch_ys).float().mean()
            accs.append(acc)

             # Log anything you want to here using logger.add_scalar
            logger.add_scalar('loss', loss_val, global_step=global_step)
            logger.add_scalar('training accuracy', acc, global_step=global_step)
            global_step += 1

            opt.zero_grad()
            loss_val.backward()
            opt.step()

        train_accs.append(torch.tensor(accs).mean().item()) 
        accs = []
        for batch_xs, batch_ys in valid_loader:
            batch_xs = batch_xs.to(device)
            batch_ys = batch_ys.to(device)
            preds = network(normalize(batch_xs))
            accs.append((preds.argmax(dim=1) == batch_ys).float().mean())
        valid_accs.append(torch.tensor(accs).mean().item())
        logger.add_scalar('validation accuracy', torch.tensor(accs).mean(), global_step=global_step)
        print("Epoch:", i, "Valid accuracy:", valid_accs[-1])

    return network, train_accs, valid_accs

In [35]:
%%time
cnn_model, cnn_train_accs, cnn_valid_accs = train(model_class=CNN, lr=5e-3, epochs=15, batch_size=128)

Epoch: 0 Valid accuracy: 0.4454113841056824
Epoch: 1 Valid accuracy: 0.5103837251663208
Epoch: 2 Valid accuracy: 0.5277887582778931
Epoch: 3 Valid accuracy: 0.5694224834442139
Epoch: 4 Valid accuracy: 0.5840585231781006
Epoch: 5 Valid accuracy: 0.6011669039726257
Epoch: 6 Valid accuracy: 0.6162974834442139
Epoch: 7 Valid accuracy: 0.6257911324501038
Epoch: 8 Valid accuracy: 0.6372626423835754
Epoch: 9 Valid accuracy: 0.6288567781448364
Epoch: 10 Valid accuracy: 0.6358781456947327
Epoch: 11 Valid accuracy: 0.6408227682113647
Epoch: 12 Valid accuracy: 0.6497231125831604
Epoch: 13 Valid accuracy: 0.6491297483444214
Epoch: 14 Valid accuracy: 0.6618868708610535
CPU times: user 1min 39s, sys: 13.1 s, total: 1min 52s
Wall time: 1min 33s


In [4]:
# Load the tensorboard 
%load_ext tensorboard
%tensorboard --logdir {log_dir} --reload_interval 1 

## Tuning Process

### Trial 1: -lr-0.001-bs-64-eps-10
For the first trial, I began with the very basic hyperparameters I've been putting as the default parameters in the training function: .001 learning rate, 10 epochs, and batch size of 64. I've also hardcoded other hyperparameters into the training code that I'm unlikely to tweak as per the tuning lecture from class, such as momentum at 0.9. This tends to be a good baseline to see if the model is working at all (the loss is decreasing, accuracy improving, etc). From here, I'm observing in the tensorboard graph that the loss is plateauing a bit--I can probably make it steeper by increasing the learning rate. 

### Trial 2: -lr-0.02-bs-64-eps-10
In this case the loss did decrease slightly more dramatically, though definitely oscillated much more at the end (as did the validation accuracy), indicating to me that I likely jumped to too large a training rate and it is struggling to settle towards the optimal value. I will decrease the learning rate slightly for the next one.

### Trial 3: -lr-0.002-bs-64-eps-10
The loss steadily decreased again this time, though it plateaued earlier than it did in trial 2 so I am going to slowly increase it over the next couple trials until it starts oscillating again.

### Trial 4: -lr-0.005-bs-64-eps-10
This learning rate has given the highest validation accuracy so far, which is a good sign. I'm going to try one more increase in learning rate, though I feel this is likely going to be around the most optimal for this model. I'd also like to note that each time I increased the learning rate the validation accuracy was higher after the first epoch (.0001 was at around 0.37, and here it was about 0.46).

### Trial 5: -lr-0.01-bs-64-eps-10
This trial technically achieved a slightly higher validation accuracy by the end, though it reintroduced the oscillation and the difference was not significant enough that I think it couldn't be reached if I simply give the model more epochs to train on the slightly smaller learning rate from trial 4. I will double the epochs in the next attempt and see how much higher the accuracy reaches. 

### Trial 6: -lr-0.005-bs-64-eps-20
The model honestly did not benefit much from doubling the epochs. The 11th epoch is the last time the accuracy was increasing before some slight oscillation occured, and I will note that it reached its highest accuracy at the 16th epoch. I will train 15 epochs in all future trials for this reason. 

### Trial 7: -lr-0.005-bs-64-eps-15
For this trial, I decided to mess with the architecture of the CNN a little. I made the channel sizes larger---all the previous ones were working as follows: 

```
self.model = nn.Sequential(
            nn.Conv2d(3, 8, (7, 7), padding=3),
            nn.ReLU(),
            nn.Conv2d(8, 16, 3, padding=1),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.AvgPool2d(kernel_size=(4, 4)),
            nn.Flatten(),
            nn.Linear(32*4*4, 10)
        )
```
And now I've changed it to this: 

```
self.model = nn.Sequential(
            nn.Conv2d(3, 16, (7, 7), padding=3),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.AvgPool2d(kernel_size=(4, 4)),
            nn.Flatten(),
            nn.Linear(64*4*4, 10)
        )
```
Such that we went from 8 to 16 to 32 before, to now starting at 16 and going to 32 and then to 64. This appears very effective to me, easily reaching the highest validation accuracy so far at about 0.67 with minimal loss oscillation all within 15 epochs. I observed similar behavior in the tensorboard lab when tuning the MLP when I increased each of the 3 hidden layer's architectures by one power of 2. 

Next, I'm thinking that the images are quite small, so I'm not sure if a 7x7 starting kernel is too large. I want to see what happens if I make it a little smaller. 

### Trial 8: -kr- 5-lr-0.005-bs-64-eps-15
This had only a small impact, though the model did reach a slightly higher validation accuracy at the end (0.68 instead of 0.67). This might just be the randomness in the training process so I think I'll run it again to see if that holds. It would make sense to me that with this specific dataset a 5x5 starting kernel could work better, though since it is not significantly higher the 7x7 seems to be a fine place to start. 

### Trial 9: -kr-5-lr-0.005-bs-64-eps-15
This trial (running the same one again) did not outperform the 7x7 kernel by any means, so it does appear that the above trial was a result of randomness in the training process. I will now return to the 7x7 kernel. 

### Trial 10: -kr-5-lr-0.005-bs-128-eps-15 (note this should be kr 7 because I did run it with the 7x7 kernel but I messed up the automatic naming)
Changing the batch size from 64 is not always positively impactful, though in previous homeworks I've had it at 128, so I wanted to see if that would impact anything here (though I feel that I'm reaching a fairly maximized accuracy with this model without pursuing something like a scheduler for the learning rate). It did appear to more quickly achieve a minimal loss and higher accuracy, though plateaued around the same value as the smaller batch size. Perhaps if I were doing less epochs this would be useful. 

### Conclusion
The last four trials had very similar levels of loss and accuracy (between 0.66 and 0.68 validation accuracy), so I am deeming this to be the best my model is going to get in the scope of this homework. To summarize my results, from the first trial to the last trials I gained approximately 10-15% accuracy by finding a good balance for the learning rate, increasing the number of epochs being run, and changing the architecture slightly to use larger channels. The final hyperparameters I would give the model are the current channel sizes from trial 7, 5e-3 learning rate, 64 batch size, and 15 epochs. 